<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task


# ML-03 — Frame Your Lane as an ML Task

## 1. My Lane as an ML task (type)

**Freestyle Capstone Title Integration:**
*Building a Content Decay Early-Warning Score — and Testing Whether AI-Referral Signals Improve It*

This is formally framed as a **binary classification problem whose calibrated probability outputs are used to construct a ranked, tier-based priority queue**—not a learning-to-rank formulation (such as pairwise or listwise loss), since the target label (`is_declining`) is evaluated per-page rather than via relative pairwise preferences. The model generates an Early Warning Risk Score (0 to 1) for every content page, which is then sorted to assist content editors in prioritizing pages requiring immediate multi-channel intervention, while functioning within an ablation framework to rigorously test the predictive lift of AI-assistant referral features.

## 2. Target or proxy

For this exploratory stage using the 53-column starter slice (`content_refresh_anonymized (1).csv`), we utilize the engineered proxy target column **`is_declining`** (derived from historical `trend_direction`). This serves as a rule-based proxy for content decay.

*Note on Data Leakage:* Because `trend_direction` (and consequently `is_declining`) has not yet been fully audited for its computation window, potential leakage between this proxy target and engineered features will be explicitly checked and addressed in the next notebook before finalizing the target definition. For production, this target maps to predicting a future multi-channel traffic drop within a subsequent time window.

## 3. Success metric

The primary operational evaluation metric is **Precision@50**, because editorial teams have limited bandwidth, making the cleanliness of the absolute top of the queue critical. However, to account for varying portfolio sizes and ensure comprehensive evaluation, **Precision@50 will be reported alongside Recall@50 and a scale-invariant Precision@top-K%** (e.g., top 1% or 5%) to capture completeness and handle diverse client portfolio scales. Additionally, ablation performance delta (comparing models with and without AI features) will be tracked via ROC-AUC.

## 4. The unit of analysis, as a real dataframe


In [ ]:
import pandas as pd

# Load the updated 53-column anonymized dataset
df = pd.read_csv("content_refresh_anonymized (1).csv")

print("Dataset shape:", df.shape)

print("\nUnit of analysis:")
print("One row represents one unique content page (content_id).")

# Display key columns including traditional search, engagement, and decline indicators
df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "trend_direction",
    "is_declining"
]].head()

Dataset shape: (7643, 53)

Unit of analysis:
One row represents one unique content page (content_id).


,content_id,client_id,impressions_90d,sessions_90d,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,3803,17,down,true
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down,true
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down,true
3,content_331d6c4de07b,client_19581e27de,11751,78,stable,false
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down,true


In this dataset, **one row represents one pseudonymized content page (`content_id`)**. Each row integrates multi-channel engagement metrics, search visibility signals, and proxy target indicators. These observable signals collectively feed into the early warning risk-scoring model for the ablation experiments.

## 5. Why ML beats a fixed rule here

A static rule (e.g., *"if impressions drop by X% and age is > 1 year, flag for review"*) struggles to capture non-linear, multi-dimensional shifts across channels. Specifically, when factoring in **AI-assistant referral traffic (ChatGPT, Claude, Gemini, Copilot, Perplexity)** alongside traditional search metrics and user engagement, the interactions become too complex for manual thresholds. Machine learning models these complex multi-channel dynamics simultaneously, producing a calibrated risk score that static single-threshold rules are unlikely to replicate as signal volume grows.

*Empirical Validation Plan:* This claim will be empirically tested in a later notebook via a multi-architecture ablation study, comparing models incorporating full feature sets against restricted baselines to evaluate whether AI referral signals provide measurable predictive lift.

## Self-check
Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.